# Buscador de Contratos SECOP-II — Búsqueda Nacional con Filtro de Categoría UNSPSC

**Documentación metodológica del script `buscar_contratos_secop_trio.py`**

Equipo de Costeo PDET — ART

---

## Objetivo del documento

Este notebook explica, fase por fase, el funcionamiento del script
**`buscar_contratos_secop_trio.py`**, responsable de la búsqueda de contratos
relacionados en SECOP-II para un conjunto de indicadores PDET. La versión aquí
documentada corresponde a la reformulación **v3**: búsqueda **nacional** (sin
restricción geográfica) combinada con un **filtro estricto de categoría
UNSPSC** (`codigo_de_categoria_principal`), en lugar de depender únicamente de
palabras clave sobre el objeto del contrato.

> **⚠ Generalización de la base de entrada**
>
> El pipeline **no está atado a los 3 indicadores del piloto**. Cualquier base
> de indicadores puede alimentarlo, siempre que sea un archivo Excel que
> contenga, como mínimo, dos columnas:
>
> - una columna con el **código del indicador** (ej. `P1.26`, `P3.14`), y
> - una columna con el **nombre / descripción del indicador**.
>
> Estas dos columnas se apuntan mediante las variables `EXCEL_INDICADORES`,
> `HOJA_INDICADORES`, `COL_CODIGO` y `COL_DESCRIPCION` (celda de
> configuración más abajo). No es necesario que la base traiga ninguna otra
> columna: la clasificación UNSPSC (segmento) y las keywords de búsqueda se
> generan o se cargan por separado y se enlazan por el código del indicador.

## Panorama general del pipeline

```
Base general de indicadores (código + nombre)
        │
        ├──► Carga de clasificación UNSPSC por indicador ──┐
        │                                                    ▼
        └──► Generación / caché de keywords (LLM) ──► Construcción del WHERE
                                                              │
                                                              ▼
                                              Cliente SECOP-II (reintentos)
                                                              │
                                                              ▼
                                        Persistencia SQLite (contratos, inventario, log)
                                                              │
                                                              ▼
                                              Exportación a Excel (8 hojas)
```

El flujo tiene tres entradas independientes que confluyen en la construcción
de la consulta SECOP: la base general de indicadores, la clasificación UNSPSC
(segmento) y las keywords de búsqueda. Todo se referencia por el **código del
indicador**, lo que permite reemplazar cualquiera de las tres piezas sin tocar
las demás.

## Control global de warnings y errores

A diferencia de un documento Quarto (que declara esto en el YAML), en un
notebook de Jupyter este control se hace en una celda de configuración al
inicio — y sigue siendo **modificable celda por celda**, ya sea cambiando las
variables globales de abajo o sobreescribiendo `warnings.filterwarnings(...)`
dentro de una celda puntual.

- `MOSTRAR_WARNINGS = False` → oculta warnings de Python (ej. `FutureWarning`
  de pandas/requests) en todo el notebook. Poner `True` para depurar.
- `DETENER_EN_ERROR = False` → las funciones de este notebook capturan sus
  propios errores de negocio y los registran en el log (nunca los corrigen en
  silencio); esta bandera solo afecta si se quiere que una excepción
  inesperada detenga la ejecución de la celda actual o solo se reporte.

In [ ]:
# ── Control global de warnings y errores (modificable por celda) ────────
import warnings

MOSTRAR_WARNINGS = False   # -> True para ver warnings de pandas/requests aquí
DETENER_EN_ERROR = False   # -> True para propagar errores inesperados en vez de solo loguearlos

if MOSTRAR_WARNINGS:
    warnings.filterwarnings("default")
else:
    warnings.filterwarnings("ignore")

# Para reactivar warnings SOLO en una celda puntual (sin afectar el resto):
#   with warnings.catch_warnings():
#       warnings.filterwarnings("default")
#       ... código a depurar ...

## Configuración global

Toda la configuración editable vive en las siguientes celdas, agrupada explícitamente para que quien use el notebook sepa que **solo ahí** debe hacer cambios.

In [ ]:
# --- Selección de proveedor de LLM (para generar keywords) --------------
import os

PROVEEDOR = os.environ.get("PDET_PROVEEDOR", "claude")
MODELO    = os.environ.get("PDET_MODELO", "claude-sonnet-4-6")

# Alternativas (descomentar según el proveedor a usar):
# PROVEEDOR = "openai";   MODELO = "gpt-5"
# PROVEEDOR = "deepseek"; MODELO = "deepseek-chat"
# PROVEEDOR = "custom";   MODELO = "llama-3.3-70b"
#   BASE_URL    = "https://api.groq.com/openai/v1"
#   API_KEY_ENV = "CUSTOM_API_KEY"

BASE_URL    = os.environ.get("PDET_BASE_URL")
API_KEY_ENV = os.environ.get("PDET_API_KEY_ENV")

In [ ]:
# ── BASE DE ENTRADA (GENERAL) ────────────────────────────────────────────
# Cualquier Excel que tenga estas dos columnas sirve como entrada:
#   - COL_CODIGO:      código único del indicador (ej. "P1.26")
#   - COL_DESCRIPCION: nombre / descripción del indicador
EXCEL_INDICADORES = "Indicadores_muestra.xlsx"   # <- reemplazar por la base propia
HOJA_INDICADORES  = "Indicadores"                # <- hoja donde están las 2 columnas
COL_CODIGO        = "codigo_indicador"
COL_DESCRIPCION   = "descripcion_indicador"

# Archivo con la clasificación UNSPSC por indicador (segmento), producido por
# un script previo de clasificación (patrón de nombre, no ruta fija):
PATRON_CLASIFICADOS = "indicadores_clasificados_trio_*.xlsx" 

In [ ]:
# ── Parámetros de la consulta a SECOP-II ─────────────────────────────────
DB_PATH      = "secop_trio.db"
EXCEL_SALIDA = "secop_trio_resultado.xlsx"
LOG_FILE     = "secop_trio_log.log"

SOCRATA_TOKEN = os.environ.get("SOCRATA_APP_TOKEN", "")
SECOP_URL     = "https://www.datos.gov.co/resource/jbjy-vk9h.json"

ANNO_MINIMO       = 2018
ESTADOS_EXCLUIDOS = ["Cancelado"]
MAX_RESULTADOS    = 150
PAUSA             = 2.0     # segundos entre llamadas (cortesía con la API)
TIMEOUT           = 120
MAX_REINTENTOS    = 3
TEMPERATURE       = 0

# Traduce el NOMBRE de segmento (texto, tal como viene de la clasificación)
# al CÓDIGO numérico UNSPSC que exige el campo real de SECOP-II
# (`codigo_de_categoria_principal`). Se agrega una entrada por cada
# segmento nuevo que se necesite filtrar (ver la última sección del notebook).
SEGMENTO_A_CODIGO_UNSPSC = {
    "Servicios de Edificación, Construcción de Instalaciones y Mantenimiento": "72",
}

CAMPOS_SELECT = (
    "id_contrato,proceso_de_compra,referencia_del_contrato,"
    "nombre_entidad,nit_entidad,departamento,ciudad,sector,"
    "tipo_de_contrato,modalidad_de_contratacion,objeto_del_contrato,"
    "valor_del_contrato,fecha_de_firma,fecha_de_fin_del_contrato,"
    "estado_contrato,proveedor_adjudicado,es_pyme,espostconflicto,"
    "urlproceso,codigo_de_categoria_principal,"
    "presupuesto_general_de_la_nacion_pgn,"
    "sistema_general_de_regal_as,sistema_general_de_participaciones,"
    "duraci_n_del_contrato"
)

### Logging

Toda advertencia o error de negocio (segmento no mapeado, indicador sin keywords, timeout de la API, etc.) se registra aquí — **nunca se corrige en silencio**: queda en el log y, adicionalmente, en la hoja `08_Log` del Excel de salida, para que el equipo metodológico (Guillermo) decida qué hacer con cada caso.

In [ ]:
import sys
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)

## Fase 1 — Capa de proveedores de LLM

Esta capa abstrae **qué modelo genera las keywords de búsqueda**, para que
cada persona del equipo pueda usar el proveedor que tenga disponible
(Anthropic, OpenAI, DeepSeek o cualquier endpoint compatible con OpenAI) sin
tocar el resto del notebook.

In [ ]:
class BaseProvider:
    label = "base"
    def complete(self, prompt: str, max_tokens: int) -> str:
        raise NotImplementedError


class AnthropicProvider(BaseProvider):
    label = "claude"
    def __init__(self, model):
        import anthropic
        key = os.environ.get("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("Falta ANTHROPIC_API_KEY en el entorno.")
        self.client = anthropic.Anthropic(api_key=key)
        self.model = model

    def complete(self, prompt, max_tokens):
        msg = self.client.messages.create(
            model=self.model, max_tokens=max_tokens, temperature=TEMPERATURE,
            messages=[{"role": "user", "content": prompt}],
        )
        return msg.content[0].text


class OpenAICompatibleProvider(BaseProvider):
    def __init__(self, label, model, api_key_env, base_url=None):
        import openai
        key = os.environ.get(api_key_env)
        if not key:
            raise RuntimeError(f"Falta {api_key_env} en el entorno.")
        self.label = label
        self.client = openai.OpenAI(api_key=key, base_url=base_url)
        self.model = model

    def complete(self, prompt, max_tokens):
        resp = self.client.chat.completions.create(
            model=self.model, max_tokens=max_tokens, temperature=TEMPERATURE,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.choices[0].message.content


def build_provider():
    if PROVEEDOR == "claude":
        return AnthropicProvider(model=MODELO)
    if PROVEEDOR == "openai":
        return OpenAICompatibleProvider("openai", MODELO, "OPENAI_API_KEY")
    if PROVEEDOR == "deepseek":
        return OpenAICompatibleProvider("deepseek", MODELO, "DEEPSEEK_API_KEY",
                                         base_url="https://api.deepseek.com")
    if PROVEEDOR == "custom":
        if not BASE_URL:
            raise RuntimeError("PROVEEDOR='custom' requiere BASE_URL en la CONFIGURACIÓN.")
        return OpenAICompatibleProvider("custom", MODELO, API_KEY_ENV or "CUSTOM_API_KEY",
                                         base_url=BASE_URL)
    raise RuntimeError(f"PROVEEDOR desconocido: '{PROVEEDOR}'")

Cada proveedor implementa el mismo contrato (`complete(prompt, max_tokens)`), por lo que el resto del pipeline es agnóstico al modelo usado. Esto sigue el patrón ya establecido en el equipo: **un solo modelo por persona**, controlado por variables de entorno, sin hardcodear llaves en el código.

## Fase 2 — Carga de la base general de indicadores

In [ ]:
import pandas as pd

def cargar_indicadores() -> pd.DataFrame:
    df = pd.read_excel(EXCEL_INDICADORES, sheet_name=HOJA_INDICADORES)
    df = df[[COL_CODIGO, COL_DESCRIPCION]].copy()
    df[COL_CODIGO] = df[COL_CODIGO].astype(str).str.strip()
    df[COL_DESCRIPCION] = df[COL_DESCRIPCION].astype(str).str.strip()
    print(f"  OK: {len(df)} indicador(es) cargado(s) de {EXCEL_INDICADORES}")
    return df

Esta función es intencionalmente mínima: solo exige que existan las dos
columnas declaradas arriba. **No asume nada más** sobre la base — no requiere
subregión, meta, ni ninguna otra columna — porque en esta fase el único
propósito es tener el par (código, nombre) para generar keywords y enlazar la
clasificación UNSPSC. Cualquier base de indicadores del proyecto (no solo el
trío piloto) puede usarse aquí con solo ajustar las 4 variables de la celda de
configuración.

## Fase 3 — Carga de la clasificación UNSPSC por indicador

In [ ]:
import glob
from typing import Dict

def cargar_categoria_por_indicador() -> Dict[str, dict]:
    candidatos = glob.glob(PATRON_CLASIFICADOS)
    resultado = {}

    if not candidatos:
        print(f"  ⚠ No se encontró ningún archivo '{PATRON_CLASIFICADOS}'.")
        print("    Corre primero el script de clasificación, o la búsqueda")
        print("    quedará SIN filtro de categoría (solo palabras clave).")
        return resultado

    preferido = f"indicadores_clasificados_trio_{PROVEEDOR}.xlsx"
    archivo = preferido if preferido in candidatos else max(candidatos, key=os.path.getmtime)
    print(f"  Usando clasificación de: {archivo}")
    df = pd.read_excel(archivo, sheet_name="Clasificacion")

    for _, row in df.iterrows():
        cod = str(row["Codigo_indicador"]).strip()
        segmento = str(row["Segmento"]).strip()
        codigo_unspsc = SEGMENTO_A_CODIGO_UNSPSC.get(segmento)
        resultado[cod] = {"segmento_nombre": segmento, "segmento_codigo": codigo_unspsc}
        if codigo_unspsc is None:
            print(f"  ⚠ Segmento '{segmento}' (indicador {cod}) no está en SEGMENTO_A_CODIGO_UNSPSC.")
            print(f"    Ese indicador se buscará SIN filtro de categoría UNSPSC (solo keywords).")

    return resultado

> **Por qué se filtra por SEGMENTO y no por Producto**
>
> Se probó restringir por el nivel más específico de la jerarquía UNSPSC
> ("Producto") y el resultado fue pobre: contratos de mejoramiento de
> vivienda o de centros de salud casi nunca están etiquetados con un código
> de producto específico en SECOP-II — quedan a nivel de familia/clase, o
> incluso con códigos administrativos genéricos de "prestación de servicios
> profesionales" aunque el objeto real sea de construcción. Filtrar solo por
> palabras clave (sin categoría) tampoco funciona: para la palabra "vivienda"
> las tres categorías más frecuentes resultaron ser un médico, un abogado y
> un auxiliar de enfermería. Filtrar por **segmento** (ej. "72" — Servicios
> de Edificación, Construcción de Instalaciones y Mantenimiento) sí
> funcionó: los resultados pasaron a ser obra de mejoramiento de vivienda
> rural, vivienda nueva y adecuación de puestos de salud.

Si un indicador nuevo requiere un segmento que aún no está en
`SEGMENTO_A_CODIGO_UNSPSC`, el indicador simplemente se busca sin filtro de
categoría (solo keywords) y queda registrado el aviso — no se detiene el
pipeline por eso.

## Fase 4 — Generación (o carga en caché) de keywords de búsqueda

In [ ]:
import json
import time

def generar_keywords(provider, nombre_indicador: str, cod_indicador: str) -> Dict:
    prompt = f"""Eres un experto en contratacion publica colombiana y politica social PDET.
Para el indicador: "{nombre_indicador}" (codigo: {cod_indicador})

Genera keywords de busqueda para encontrar contratos relacionados en SECOP II,
a NIVEL NACIONAL (sin restriccion geografica).
Responde SOLO con JSON, sin texto adicional, sin bloques de codigo:
{{
  "kw_primarias": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "kw_secundarias": ["kw6", "kw7", "kw8", "kw9", "kw10"]
}}

Reglas:
- Sin tildes ni caracteres especiales (la API SECOP las rechaza)
- Palabras clave tecnicas del sector publico colombiano
- Primarias: las mas especificas y discriminantes del PRODUCTO/SERVICIO exacto
- Secundarias: terminos relacionados mas amplios (programas, entidades tipicas)
"""
    try:
        raw = provider.complete(prompt, max_tokens=400)
        raw = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(raw)
    except Exception as e:
        logging.warning(f"Error generando keywords [{provider.label}] para {cod_indicador}: {e}")
        return keywords_fallback(nombre_indicador)


def keywords_fallback(nombre_indicador: str) -> Dict:
    """Si el LLM falla, se generan keywords simples a partir del propio
    nombre del indicador — nunca se deja el indicador sin ninguna keyword."""
    limpio = (nombre_indicador.replace("á", "a").replace("é", "e").replace("í", "i")
              .replace("ó", "o").replace("ú", "u").replace("ñ", "n"))
    palabras = [p for p in limpio.split() if len(p) > 4][:5]
    return {"kw_primarias": palabras if palabras else ["contrato", "servicio"],
            "kw_secundarias": ["rural", "PDET", "comunidad", "beneficiario"]}


def cargar_o_generar_keywords(df_ind: pd.DataFrame, provider, dry_run: bool,
                               json_path: str = "keywords_trio.json") -> Dict:
    cache = {}
    if os.path.exists(json_path):
        try:
            cache = json.load(open(json_path, "r", encoding="utf-8"))
            print(f"  Keywords cargadas del cache: {len(cache)} indicadores")
        except Exception:
            pass

    faltantes = [c for c in df_ind[COL_CODIGO] if c not in cache]
    if not faltantes:
        return cache

    if dry_run or provider is None:
        print(f"  [dry-run / sin proveedor] Usando keywords genéricas para {len(faltantes)} indicadores.")
        for cod in faltantes:
            nombre = df_ind.loc[df_ind[COL_CODIGO] == cod, COL_DESCRIPCION].iloc[0]
            cache[cod] = {"nombre": nombre, **keywords_fallback(nombre)}
        return cache

    print(f"  Generando keywords con {provider.label} ({MODELO}) para {len(faltantes)} indicadores...")
    for i, cod in enumerate(faltantes, 1):
        nombre = df_ind.loc[df_ind[COL_CODIGO] == cod, COL_DESCRIPCION].iloc[0]
        print(f"    [{i}/{len(faltantes)}] {cod}: {nombre[:60]}...")
        kws = generar_keywords(provider, nombre, cod)
        cache[cod] = {"nombre": nombre, **kws}
        time.sleep(0.3)

    json.dump(cache, open(json_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"  Keywords guardadas en: {json_path}")
    return cache

El caché (`keywords_trio.json`) evita volver a llamar al LLM para indicadores ya procesados — clave para el patrón de scripts **reanudables** del equipo. Si no hay proveedor disponible (`dry_run=True` o sin llave), se usa el *fallback* determinístico basado en el propio nombre del indicador, para que el pipeline nunca quede bloqueado por falta de credenciales.

## Fase 5 — Construcción de la consulta SECOP (cláusulas `WHERE`)

In [ ]:
from typing import List

def clause_estado_excluir(excluidos: List[str]) -> str:
    excluidos_lower = [f"'{e.lower()}'" for e in excluidos]
    return f"lower(estado_contrato) NOT IN ({', '.join(excluidos_lower)})"


def clause_anno() -> str:
    return f"fecha_de_firma >= '{ANNO_MINIMO}-01-01T00:00:00.000'"


def clause_keywords(kw_lista: List[str]) -> str:
    """
    Cada keyword puede ser una frase de varias palabras. En SECOP-II el
    texto real casi siempre trae conectores entre ellas ("mejoramiento DE
    vivienda rural"), así que exigir la frase completa como substring
    literal da 0 resultados aunque el contrato sea relevante. Por eso cada
    keyword se descompone en palabras (máx. 2, las más sustantivas) y se
    exige que TODAS aparezcan en el objeto del contrato, en cualquier orden.
    Las distintas keywords se combinan entre sí con OR.
    """
    grupos = []
    for kw in kw_lista[:8]:
        palabras = [p for p in kw.lower().split() if len(p) > 2][:2]
        if not palabras:
            continue
        grupo = " AND ".join(f"objeto_del_contrato LIKE '%{p}%'" for p in palabras)
        grupos.append(f"({grupo})")
    return "(" + " OR ".join(grupos) + ")" if grupos else "(1=1)"


def clause_categoria(segmento_codigo: str) -> str:
    return f"codigo_de_categoria_principal LIKE 'V1.{segmento_codigo}%'"


def build_query(kw_prim: List[str], kw_sec: List[str], segmento_codigo: str) -> str:
    kws_all = list(dict.fromkeys(kw_prim + kw_sec))
    partes = [clause_keywords(kws_all), clause_estado_excluir(ESTADOS_EXCLUIDOS), clause_anno()]
    if segmento_codigo:
        partes.insert(1, clause_categoria(segmento_codigo))
    return " AND ".join(partes)

Nótese que la cláusula de categoría solo se agrega **si el indicador tiene segmento mapeado** (Fase 3); si no lo tiene, la búsqueda cae de forma controlada a solo-keywords, y ese hecho queda documentado en el inventario final (columna `fuente_ajuste` / `nota_ajuste`).

## Fase 6 — Cliente SECOP-II (API Socrata)

In [ ]:
import requests

class SECOPClient:
    def __init__(self, token: str = None):
        headers = {"Accept": "application/json"}
        if token:
            headers["X-App-Token"] = token
            print("  Token Socrata configurado.")
        else:
            print("  AVISO: Sin token Socrata (SOCRATA_APP_TOKEN). Límite ~1000 consultas/hora.")
        self.session = requests.Session()
        self.session.headers.update(headers)

    def query(self, where: str, limit: int = MAX_RESULTADOS) -> List[Dict]:
        params = {"$select": CAMPOS_SELECT, "$where": where, "$limit": limit,
                   "$order": "fecha_de_firma DESC"}
        for intento in range(1, MAX_REINTENTOS + 1):
            try:
                r = self.session.get(SECOP_URL, params=params, timeout=TIMEOUT)
                if r.status_code == 400:
                    logging.error(f"Error 400 Bad Request. Query: {where[:300]}")
                    return []
                r.raise_for_status()
                return r.json()
            except requests.exceptions.ReadTimeout:
                espera = 10 * intento
                print(f"    ⏱ Timeout intento {intento}/{MAX_REINTENTOS} — esperando {espera}s")
                time.sleep(espera)
            except requests.exceptions.ConnectionError:
                espera = 15 * intento
                print(f"    ⚠ Error de conexión intento {intento}/{MAX_REINTENTOS} — esperando {espera}s")
                time.sleep(espera)
            except requests.exceptions.HTTPError as e:
                logging.warning(f"HTTP Error: {e}")
                return []
            except Exception as e:
                logging.warning(f"Error inesperado: {e}")
                time.sleep(5)
        print(f"    ✗ {MAX_REINTENTOS} intentos fallidos — se omite esta consulta")
        return []

El cliente reintenta ante *timeouts* y errores de conexión con espera creciente (`10 * intento`, `15 * intento`), pero **nunca reintenta** un error 400 (query mal formada) — ese caso se registra como error y se devuelve una lista vacía de inmediato, porque reintentar una consulta inválida no cambiará el resultado.

## Fase 7 — Persistencia en SQLite

In [ ]:
import sqlite3

def init_db(db_path: str) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS inventario (
            id                INTEGER PRIMARY KEY AUTOINCREMENT,
            subregion         TEXT NOT NULL,
            cod_indicador     TEXT NOT NULL,
            nombre_indicador  TEXT,
            departamento_sr   TEXT,
            total_c1          INTEGER DEFAULT 0,
            total_c2          INTEGER DEFAULT 0,
            total_c3_otras_sr INTEGER DEFAULT 0,
            total_c4_nacional INTEGER DEFAULT 0,
            total_contratos   INTEGER DEFAULT 0,
            annos_lista       TEXT,
            anno_min          INTEGER,
            anno_max          INTEGER,
            valor_total       REAL DEFAULT 0,
            valor_promedio    REAL DEFAULT 0,
            valor_min         REAL DEFAULT 0,
            valor_max         REAL DEFAULT 0,
            entidades_lista   TEXT,
            nivel_confianza   TEXT,
            hay_datos_propios INTEGER DEFAULT 0,
            requiere_ajuste   INTEGER DEFAULT 0,
            fuente_ajuste     TEXT,
            nota_ajuste       TEXT,
            sin_datos_contratos INTEGER DEFAULT 0,
            keywords_usadas   TEXT,
            timestamp         TEXT,
            UNIQUE(subregion, cod_indicador)
        );

        CREATE TABLE IF NOT EXISTS contratos (
            id_contrato       TEXT NOT NULL,
            cod_indicador     TEXT NOT NULL,
            subregion         TEXT NOT NULL,
            capa              INTEGER NOT NULL,
            referencia        TEXT,
            proceso_compra    TEXT,
            nombre_entidad    TEXT,
            nit_entidad       TEXT,
            departamento      TEXT,
            ciudad            TEXT,
            sector            TEXT,
            tipo_contrato     TEXT,
            modalidad         TEXT,
            objeto_contrato   TEXT,
            categoria_unspsc  TEXT,
            valor_contrato    REAL DEFAULT 0,
            anno_firma        INTEGER,
            fecha_firma       TEXT,
            fecha_fin         TEXT,
            estado_contrato   TEXT,
            proveedor         TEXT,
            es_pyme           TEXT,
            espostconflicto   TEXT,
            url_proceso       TEXT,
            fuente_pgn        REAL DEFAULT 0,
            fuente_sgr        REAL DEFAULT 0,
            fuente_sgp        REAL DEFAULT 0,
            duracion          TEXT,
            subregion_origen  TEXT,
            fecha_consulta    TEXT,
            validado_llm      INTEGER DEFAULT NULL,
            PRIMARY KEY(id_contrato, cod_indicador, subregion, capa)
        );

        CREATE TABLE IF NOT EXISTS log_queries (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            subregion     TEXT,
            cod_indicador TEXT,
            capa          INTEGER,
            n_resultados  INTEGER,
            where_clause  TEXT,
            timestamp     TEXT
        );
    """)
    conn.commit()

    # Migración defensiva: si "contratos" ya existía de una corrida anterior
    # con un esquema más viejo, agrega la columna que falte en vez de tronar.
    columnas_actuales = [r[1] for r in conn.execute("PRAGMA table_info(contratos)").fetchall()]
    if "categoria_unspsc" not in columnas_actuales:
        conn.execute("ALTER TABLE contratos ADD COLUMN categoria_unspsc TEXT")
        conn.commit()

    return conn

Tres tablas: **`inventario`** (una fila por indicador, con el resumen estadístico de la búsqueda), **`contratos`** (el detalle de cada contrato encontrado, con `validado_llm` reservado para la validación semántica posterior) y **`log_queries`** (auditoría de cada consulta ejecutada contra SECOP-II, con la cláusula `WHERE` completa). El bloque de migración defensiva permite reejecutar el notebook sobre una base de datos ya existente de una versión anterior sin perder los datos ya cargados.

In [ ]:
from datetime import datetime

def guardar_contratos_db(conn, contratos, cod_ind, subregion="NACIONAL", capa=1):
    ts = datetime.now().isoformat()
    cur = conn.cursor()
    for c in contratos:
        idc = c.get("id_contrato", "")
        if not idc:
            continue
        urlproceso = c.get("urlproceso")
        url = urlproceso.get("url", "") if isinstance(urlproceso, dict) else ""
        anno = None
        fecha = c.get("fecha_de_firma", "")
        if fecha:
            try:
                anno = int(str(fecha)[:4])
            except Exception:
                pass
        cur.execute("""
            INSERT OR REPLACE INTO contratos (
                id_contrato, cod_indicador, subregion, capa,
                referencia, proceso_compra, nombre_entidad, nit_entidad,
                departamento, ciudad, sector, tipo_contrato, modalidad,
                objeto_contrato, categoria_unspsc, valor_contrato,
                anno_firma, fecha_firma, fecha_fin, estado_contrato,
                proveedor, es_pyme, espostconflicto, url_proceso,
                fuente_pgn, fuente_sgr, fuente_sgp, duracion,
                subregion_origen, fecha_consulta
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            idc, cod_ind, subregion, capa,
            c.get("referencia_del_contrato", ""), c.get("proceso_de_compra", ""),
            c.get("nombre_entidad", ""), c.get("nit_entidad", ""),
            c.get("departamento", ""), c.get("ciudad", ""), c.get("sector", ""),
            c.get("tipo_de_contrato", ""), c.get("modalidad_de_contratacion", ""),
            c.get("objeto_del_contrato", ""), c.get("codigo_de_categoria_principal", ""),
            float(c.get("valor_del_contrato", 0) or 0),
            anno, fecha, c.get("fecha_de_fin_del_contrato", ""), c.get("estado_contrato", ""),
            c.get("proveedor_adjudicado", ""), c.get("es_pyme", ""), c.get("espostconflicto", ""),
            url, float(c.get("presupuesto_general_de_la_nacion_pgn", 0) or 0),
            float(c.get("sistema_general_de_regal_as", 0) or 0),
            float(c.get("sistema_general_de_participaciones", 0) or 0),
            c.get("duraci_n_del_contrato", ""), subregion, ts,
        ))
    conn.commit()

`INSERT OR REPLACE` sobre la llave compuesta `(id_contrato, cod_indicador, subregion, capa)` hace que el proceso sea **idempotente**: correrlo dos veces sobre el mismo indicador no duplica contratos, solo actualiza el registro con los datos más recientes de SECOP.

In [ ]:
def calcular_inventario(conn, cod_ind, nombre_ind, contratos, kws_str, segmento_info):
    valores = [float(c.get("valor_del_contrato", 0) or 0) for c in contratos
               if float(c.get("valor_del_contrato", 0) or 0) > 0]
    annos = []
    for c in contratos:
        f = c.get("fecha_de_firma", "")
        if f:
            try:
                annos.append(int(str(f)[:4]))
            except Exception:
                pass
    entidades = list(dict.fromkeys(
        [c.get("nombre_entidad", "") for c in contratos if c.get("nombre_entidad", "")]
    ))[:10]

    n = len(contratos)
    confianza = "Alto" if n >= 5 else ("Medio" if n >= 1 else "Sin datos")
    hay_datos = 1 if n >= 1 else 0
    sin_datos = 1 if n == 0 else 0

    seg_nombre = segmento_info.get("segmento_nombre", "?")
    seg_codigo = segmento_info.get("segmento_codigo")
    fuente = (f"Búsqueda nacional + filtro UNSPSC segmento {seg_codigo} ({seg_nombre})"
              if seg_codigo else
              f"Búsqueda nacional SIN filtro UNSPSC (segmento '{seg_nombre}' no mapeado)")

    conn.execute("""
        INSERT OR REPLACE INTO inventario (
            subregion, cod_indicador, nombre_indicador, departamento_sr,
            total_c1, total_c2, total_c3_otras_sr, total_c4_nacional,
            total_contratos, annos_lista, anno_min, anno_max,
            valor_total, valor_promedio, valor_min, valor_max,
            entidades_lista, nivel_confianza, hay_datos_propios,
            requiere_ajuste, fuente_ajuste, nota_ajuste,
            sin_datos_contratos, keywords_usadas, timestamp
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        "NACIONAL", cod_ind, nombre_ind, "Colombia (nacional, filtrado por categoría UNSPSC)",
        n, 0, 0, 0, n,
        json.dumps(sorted(set(annos))),
        min(annos) if annos else None, max(annos) if annos else None,
        sum(valores) if valores else 0,
        round(sum(valores) / len(valores), 0) if valores else 0,
        min(valores) if valores else 0, max(valores) if valores else 0,
        json.dumps(entidades), confianza, hay_datos, 0,
        fuente, f"Segmento UNSPSC objetivo: {seg_codigo or 'ninguno'} ({seg_nombre})",
        sin_datos, kws_str, datetime.now().isoformat(),
    ))
    conn.commit()


def log_query(conn, cod, n, where):
    conn.execute(
        "INSERT INTO log_queries (subregion, cod_indicador, capa, n_resultados, where_clause, timestamp) "
        "VALUES (?,?,?,?,?,?)",
        ("NACIONAL", cod, 1, n, (where or "")[:600], datetime.now().isoformat()),
    )
    conn.commit()

El **nivel de confianza** (`Alto` / `Medio` / `Sin datos`) es un umbral simple sobre el número de contratos encontrados (≥5 / ≥1 / 0). No sustituye la validación metodológica de Guillermo — es solo una señal rápida para priorizar qué indicadores revisar primero.

## Fase 8 — Exportación a Excel

In [ ]:
def exportar_excel(conn, excel_out):
    print(f"\nExportando a: {excel_out}")
    with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
        df_inv = pd.read_sql_query("""
            SELECT subregion, cod_indicador, nombre_indicador, departamento_sr,
                   nivel_confianza, hay_datos_propios, requiere_ajuste, sin_datos_contratos,
                   total_contratos, total_c1 AS contratos_subregion_exacta,
                   total_c2 AS contratos_departamento, total_c3_otras_sr AS contratos_otras_subregiones,
                   total_c4_nacional AS contratos_referencia_nacional,
                   anno_min, anno_max, annos_lista,
                   ROUND(valor_total,0) AS valor_total_cop, ROUND(valor_promedio,0) AS valor_promedio_cop,
                   ROUND(valor_min,0) AS valor_min_cop, ROUND(valor_max,0) AS valor_max_cop,
                   fuente_ajuste, nota_ajuste, entidades_lista, keywords_usadas
            FROM inventario ORDER BY cod_indicador
        """, conn)
        df_inv.to_excel(writer, sheet_name="01_Inventario_Completo", index=False)
        df_inv[df_inv["hay_datos_propios"] == 1].to_excel(writer, sheet_name="02_Con_Datos_Propios", index=False)
        df_inv[df_inv["requiere_ajuste"] == 1].to_excel(writer, sheet_name="03_Requieren_Ajuste", index=False)
        df_inv[df_inv["sin_datos_contratos"] == 1].to_excel(writer, sheet_name="04_Sin_Datos_Contratos", index=False)

        df_urls = pd.read_sql_query("""
            SELECT c.subregion, c.cod_indicador, i.nombre_indicador, c.capa,
                   'Nacional + categoria UNSPSC' AS nivel_geografico,
                   c.id_contrato, c.referencia, c.proceso_compra,
                   c.nombre_entidad, c.departamento, c.ciudad,
                   c.tipo_contrato, c.objeto_contrato, c.categoria_unspsc,
                   ROUND(c.valor_contrato,0) AS valor_contrato_cop,
                   c.anno_firma, c.estado_contrato, c.es_pyme, c.espostconflicto, c.duracion,
                   c.url_proceso
            FROM contratos c
            LEFT JOIN inventario i ON c.cod_indicador=i.cod_indicador AND c.subregion=i.subregion
            WHERE c.capa IN (1,2)
            ORDER BY c.cod_indicador, c.valor_contrato DESC
        """, conn)
        df_urls.to_excel(writer, sheet_name="05_Contratos_IDs_URLs", index=False)

        # Hojas reservadas para el flujo de ajuste territorial (se llenan en
        # una etapa posterior del pipeline, no en este notebook)
        pd.DataFrame(columns=["subregion_sin_datos", "cod_indicador", "nombre_indicador",
                               "subregion_referencia", "id_contrato", "referencia",
                               "nombre_entidad", "departamento", "ciudad", "tipo_contrato",
                               "objeto_contrato", "valor_contrato_cop", "anno_firma",
                               "estado_contrato", "url_proceso"]
                     ).to_excel(writer, sheet_name="06_Contratos_Otras_SR_Ajuste", index=False)
        pd.DataFrame(columns=["subregion", "cod_indicador", "nombre_indicador", "id_contrato",
                               "nombre_entidad", "departamento", "ciudad", "objeto_contrato",
                               "valor_contrato_cop", "anno_firma", "estado_contrato", "url_proceso"]
                     ).to_excel(writer, sheet_name="07_Contratos_Ref_Nacional", index=False)

        df_log = pd.read_sql_query("SELECT * FROM log_queries ORDER BY timestamp", conn)
        df_log.to_excel(writer, sheet_name="08_Log", index=False)

    print(f"  Guardado: {excel_out}")

El archivo de salida mantiene **el mismo formato de 8 hojas** que
versiones anteriores del script, por compatibilidad: los pasos posteriores
del pipeline (validación semántica de contratos, descarga de PDFs) leen este
archivo sin cambios.

| Hoja | Contenido |
|---|---|
| `01_Inventario_Completo` | Un resumen por indicador (todos) |
| `02_Con_Datos_Propios` | Indicadores con ≥1 contrato encontrado |
| `03_Requieren_Ajuste` | Reservada para el ajuste territorial posterior |
| `04_Sin_Datos_Contratos` | Indicadores sin ningún contrato |
| `05_Contratos_IDs_URLs` | Detalle de cada contrato (capas 1 y 2) |
| `06_Contratos_Otras_SR_Ajuste` | Reservada (se llena en otra etapa) |
| `07_Contratos_Ref_Nacional` | Reservada (se llena en otra etapa) |
| `08_Log` | Auditoría de cada consulta ejecutada contra SECOP |

## Fase 9 — Orquestación del proceso (`run_batch`, `run_test`)

In [ ]:
def run_batch(dry_run=False):
    df_ind = cargar_indicadores()
    categorias = cargar_categoria_por_indicador()

    provider = None
    if not dry_run:
        try:
            provider = build_provider()
        except RuntimeError as e:
            print(f"\n❌  {e}\n   (puedes seguir con dry_run=True mientras tanto)")
            return

    kws = cargar_o_generar_keywords(df_ind, provider, dry_run)
    client = SECOPClient(SOCRATA_TOKEN)
    conn = init_db(DB_PATH)

    total = len(df_ind)
    for i, row in df_ind.iterrows():
        cod_ind, nombre_ind = row[COL_CODIGO], row[COL_DESCRIPCION]
        ind_data = kws.get(cod_ind, {})
        kw_prim, kw_sec = ind_data.get("kw_primarias", []), ind_data.get("kw_secundarias", [])
        kws_str = json.dumps(ind_data)
        seg_info = categorias.get(cod_ind, {"segmento_nombre": "?", "segmento_codigo": None})

        print(f"\n[{i+1}/{total}] {cod_ind} | {nombre_ind[:70]}")
        print(f"  KW primarias: {kw_prim}")
        print(f"  Filtro UNSPSC: segmento {seg_info['segmento_codigo'] or '(ninguno)'} — {seg_info['segmento_nombre']}")

        q = build_query(kw_prim, kw_sec, seg_info["segmento_codigo"])
        if dry_run:
            print(f"  🔍 [dry-run] query: {q[:250]}...")
            continue

        resultados = client.query(q, limit=MAX_RESULTADOS)
        log_query(conn, cod_ind, len(resultados), q)
        guardar_contratos_db(conn, resultados, cod_ind)
        calcular_inventario(conn, cod_ind, nombre_ind, resultados, kws_str, seg_info)

        print(f"  → {len(resultados)} contratos encontrados")
        time.sleep(PAUSA)

    if not dry_run:
        exportar_excel(conn, EXCEL_SALIDA)
    conn.close()

`run_batch` es el flujo completo: carga indicadores → carga clasificación → genera/recupera keywords → construye y ejecuta la consulta por indicador → persiste en SQLite → exporta a Excel. El modo `dry_run=True` corre todo el flujo **sin llamar a la API de SECOP ni al LLM**, solo mostrando las queries que se construirían — útil para validar la configuración antes de una corrida real.

In [ ]:
def run_test(dry_run=False):
    """Corre sobre todos los indicadores pero con una muestra de solo 5
    contratos por indicador — pensado para validar rápido antes de un
    run_batch completo."""
    df_ind = cargar_indicadores()
    categorias = cargar_categoria_por_indicador()
    provider = None
    if not dry_run:
        try:
            provider = build_provider()
        except RuntimeError as e:
            print(f"\n❌  {e}")
            return
    kws = cargar_o_generar_keywords(df_ind, provider, dry_run)
    client = SECOPClient(SOCRATA_TOKEN)

    for _, row in df_ind.iterrows():
        cod_ind, nombre_ind = row[COL_CODIGO], row[COL_DESCRIPCION]
        ind_data = kws.get(cod_ind, {})
        kw_prim, kw_sec = ind_data.get("kw_primarias", []), ind_data.get("kw_secundarias", [])
        seg_info = categorias.get(cod_ind, {"segmento_nombre": "?", "segmento_codigo": None})
        print(f"\n{'='*60}\n{cod_ind} | {nombre_ind[:70]}")
        print(f"KW: {kw_prim} | {kw_sec}")
        print(f"Segmento UNSPSC: {seg_info['segmento_codigo']} ({seg_info['segmento_nombre']})\n{'='*60}")

        q = build_query(kw_prim, kw_sec, seg_info["segmento_codigo"])
        if dry_run:
            print("[dry-run] query construida, no se llamó a SECOP")
            continue

        resultados = client.query(q, limit=5)
        print(f"Contratos (muestra de 5 de hasta {MAX_RESULTADOS}): {len(resultados)}")
        for c in resultados[:5]:
            url = c.get("urlproceso", {})
            url = url.get("url", "") if isinstance(url, dict) else ""
            print(f"  [{c.get('codigo_de_categoria_principal','')}] {c.get('nombre_entidad','')} | {c.get('ciudad','')}")
            print(f"    {(c.get('objeto_del_contrato','') or '')[:100]}")
            print(f"    id_contrato={c.get('id_contrato','')} | URL={url[:70]}")
        time.sleep(PAUSA)

## Fase 10 — Ejecución

En el script original esto se maneja por línea de comandos (`argparse`), con
`--modo batch|test` y `--dry-run`. En un notebook no hay CLI: se llama
directamente a la función deseada, cambiando los argumentos de la celda.

Por defecto la celda siguiente queda en modo **dry-run** (no llama a SECOP ni
al LLM) para que el notebook se pueda ejecutar de principio a fin sin
credenciales. Cambiar `DRY_RUN = False` para una corrida real (requiere
`ANTHROPIC_API_KEY` / `SOCRATA_APP_TOKEN` en el entorno y la base de
indicadores real en `EXCEL_INDICADORES`).

In [ ]:
print("=" * 60)
print(f"  Búsqueda SECOP-II — nacional + categoría UNSPSC — proveedor: {PROVEEDOR} ({MODELO})")
print(f"  Estados excluidos: {ESTADOS_EXCLUIDOS}")
print("=" * 60)

DRY_RUN = True   # -> False para correr contra SECOP y el LLM de verdad
MODO    = "batch"  # -> "test" para la muestra rápida de 5 contratos por indicador

try:
    if MODO == "test":
        run_test(dry_run=DRY_RUN)
    else:
        run_batch(dry_run=DRY_RUN)
except Exception as e:
    logging.error(f"Fallo en la ejecución: {e}")
    if DETENER_EN_ERROR:
        raise

## Apéndice — Cómo descubrir el código UNSPSC de un segmento nuevo

Cuando se clasifique un indicador que caiga en un segmento distinto al ya
cargado en `SEGMENTO_A_CODIGO_UNSPSC`, se agrega consultando directamente la
API con una palabra clave representativa del indicador:

In [ ]:
import requests

r = requests.get(
    "https://www.datos.gov.co/resource/jbjy-vk9h.json",
    params={
        "$select": "codigo_de_categoria_principal, count(*) as n",
        "$where": "objeto_del_contrato LIKE '%tu_keyword%'",
        "$group": "codigo_de_categoria_principal",
        "$order": "n DESC", "$limit": 15,
    },
    headers={"Accept": "application/json"},
)
for row in r.json():
    print(row)

Los primeros dos dígitos del código con mayor conteo (`n`) son el segmento a agregar al diccionario. Este paso, al igual que cualquier otra anomalía del pipeline, debe validarse antes de incorporarse de forma definitiva a la clasificación.